In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'CROSSREF-DOI-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
crossref_file_name = 'DOITEST.txt'
use_files_tab = False
use_file_view = False
crossref_doi = '10.52825/cordi.v1i.260'
crossref_expected_title_en = 'Toward the Development of NII RDC Application Profile Using Ontology Technology'
crossref_expected_journal_en = 'Proceedings of the Conference on Research Data Infrastructure'
crossref_expected_date_published = '2023-09'
crossref_expected_volume = '1'
crossref_expected_issue = '1'
crossref_expected_page_start = '1'
crossref_expected_page_end = '8'
crossref_expected_manuscript_value = 'journal article'
crossref_expected_author_keyword = 'Minamiyama'
default_result_path = None
close_on_fail = False
transition_timeout = 10000
delete_project = True
jalc_doi = '10.2964/jsik_2023_020'
jalc_expected_title_ja = 'オントロジー技術を用いたNII RDCアプリケーションプロファイル開発に向けて'
jalc_expected_title_en = 'Toward the development of NII RDC application profile using ontology technology'
jalc_expected_journal_ja = '情報知識学会誌'
jalc_expected_journal_en = 'Journal of Japan Society of Information and Knowledge'
jalc_expected_date_published = '2023-05'
jalc_expected_volume = '33'
jalc_expected_issue = '2'
jalc_expected_page_start = '212'
jalc_expected_page_end = '220'
jalc_expected_author_keyword = 'MINAMIYAMA'
pubmed_doi = '10.1371/journal.pone.0301772'
pubmed_expected_title_en = "A study on formalizing the knowledge of data curation activities across different fields"
pubmed_expected_journal_en = "PLoS One"
pubmed_expected_date_published = '2024'
pubmed_expected_volume = '19'
pubmed_expected_issue = '4'
pubmed_expected_page_start = 'e0301772'
pubmed_expected_page_end = '220'
pubmed_expected_author_keyword = 'Minamiyama'

arxiv_doi = '10.48550/arXiv.2001.04232'
arxiv_expected_title_en = 'Possibility and prevention of inappropriate data manipulation in Polar Data Journal'
arxiv_expected_journal_en = '2019 8th International Congress on Advanced Applied Informatics (IIAI-AAI)'
arxiv_expected_date_published = '2020-01'
arxiv_expected_volume = ''
arxiv_expected_issue = ''
arxiv_expected_page_start = ''
arxiv_expected_page_end = ''
arxiv_expected_author_keyword = 'Minamiyama'

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

### DOI補完確認 (Crossref / JaLC / PubMed / arXiv)

- サブシステム名: アドオン
- ページ/アドオン: Metadata
- 機能分類: メタデータ入力
- シナリオ名: Crossref、JaLC、PubMed、arXivを利用した書誌情報の自動補完
- 用意するテストデータ: DOI (Crossref: 10.52825/cordi.v1i.260 / JaLC: 10.2964/jsik_2023_020 / PubMed: 10.1371/journal.pone.0301772 / arXiv: 10.48550/arXiv.2001.04232)、URL一覧、アカウント(既存ユーザー1: GRDM)
- 事前条件: なし

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)
import scripts.metadata_v2025
importlib.reload(scripts.metadata_v2025)

from scripts.playwright import *
from scripts import grdm
from scripts.metadata_v2025 import FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から指定されたプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    if use_files_tab:
        await page.locator('//*[@id = "projectNavFiles"]//a[contains(text(), "ファイル")]').click()
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージをクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    await grdm.get_select_expanded_storage_title_locator(page, target_storage_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージにテスト用ファイルをアップロードする

ファイルがアップロードされ、ファイル一覧に表示されること

In [ ]:
import os
import asyncio

# テスト用ファイルの作成
crossref_file_path = os.path.join(work_dir, crossref_file_name)
with open(crossref_file_path, 'w') as f:
    f.write('DOI Test File')

async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, crossref_file_path)
    await asyncio.sleep(1)

    await expect(page.locator(f'//*[text() = "{crossref_file_name}"]/../following-sibling::*//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)
    file_locator = grdm.get_select_file_title_locator(page, crossref_file_name)
    await expect(file_locator).to_be_visible(timeout=transition_timeout)
    await file_locator.scroll_into_view_if_needed()

await run_pw(_step)

## テスト用ファイルをクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    if use_file_view:
        await grdm.get_select_file_title_locator(page, crossref_file_name).click()
    else:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックする

「メタデータ編集」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('ファイル種別')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ファイル種別」で「論文」を選択する

論文専用項目が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('ファイル種別')
    await locator.select_option('manuscript')
    await expect(locator).to_have_value('manuscript', timeout=transition_timeout)

await run_pw(_step)

## 「論文（出版社版）のDOI」にテスト用DOIを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('論文（出版社版）のDOI', crossref_doi)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_have_value(crossref_doi, timeout=transition_timeout)

await run_pw(_step)

## 「Crossrefから取得」ボタンをクリックする

ボタン押下後、自動補完が完了すること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    button = page.locator('//label[contains(text(), "論文（出版社版）のDOI")]/../following-sibling::div[1]//button[.//span[contains(text(), "Crossrefから取得")]]')
    await expect(button).to_be_enabled(timeout=transition_timeout)
    await button.click()

    await expect(form.get_locator('Title (English)')).to_have_value(
        crossref_expected_title_en,
        timeout=transition_timeout,
    )

await run_pw(_step)

## 自動補完された項目を確認する

- Title (English)
- Journal Name (English)
- 発行年月
- 巻・号
- 掲載ページ (開始/終了)
- 論文の種類
- 著者名 (英語表記)

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(crossref_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(crossref_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(crossref_expected_date_published, timeout=transition_timeout)
    await expect(form.get_locator('巻')).to_have_value(crossref_expected_volume, timeout=transition_timeout)
    await expect(form.get_locator('号')).to_have_value('', timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (開始)')).to_have_value('', timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (終了)')).to_have_value('', timeout=transition_timeout)
    await expect(form.get_locator('論文の種類')).to_have_value(crossref_expected_manuscript_value, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(crossref_expected_author_keyword, timeout=transition_timeout)

await run_pw(_step)

## 自動補完されない項目が空欄のままであることを確認する

- データの名称または論文表題 (日本語)
- 掲載誌名 (日本語)

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await expect(form.get_locator('データの名称または論文表題 (日本語)')).to_have_value('', timeout=transition_timeout)
    await expect(form.get_locator('掲載誌名 (日本語)')).to_have_value('', timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックする

ダイアログが閉じること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//a[text() = "保存"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、メタデータ編集ボタンを再度表示する

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、自動補完された値が保持されていることを確認する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(crossref_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(crossref_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(crossref_expected_date_published, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(crossref_expected_author_keyword, timeout=transition_timeout)

    await page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]').click()
    await expect(page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、JaLC補完の準備をする

JaLC補完でも同じファイルを利用するため、再度ファイルを選択し「メタデータ編集」ボタンを有効にする。

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、JaLC補完を開始する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「論文（出版社版）のDOI」にJaLCのDOIを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('論文（出版社版）のDOI', jalc_doi)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_have_value(jalc_doi, timeout=transition_timeout)

await run_pw(_step)

## 「JaLCから取得」を選択して補完する

「論文（出版社版）のDOI」欄右側の▼をクリックし、表示されたメニューから「JaLCから取得」を選択する。押下後、JaLCから取得したメタデータに置き換わることを確認する。

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = page.locator('//label[contains(text(), "論文（出版社版）のDOI")]/../following-sibling::div[1]')
    dropdown_button = container.locator('button.dropdown-toggle')
    await dropdown_button.click()
    menu_item = container.locator('//ul[contains(@class, "dropdown-menu")]//a[text() = "JaLCから取得"]')
    await expect(menu_item).to_be_visible(timeout=transition_timeout)
    await menu_item.click()

    await expect(form.get_locator('データの名称または論文表題 (日本語)')).to_have_value(
        jalc_expected_title_ja, timeout=transition_timeout
    )

await run_pw(_step)

## JaLCで自動補完された項目を確認する

- データの名称または論文表題 (日本語)
- Title (English)
- 掲載誌名 (日本語)
- Journal Name (English)
- 発行年月
- 巻・号
- 掲載ページ (開始/終了)
- 著者名

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await expect(form.get_locator('データの名称または論文表題 (日本語)')).to_have_value(jalc_expected_title_ja, timeout=transition_timeout)
    await expect(form.get_locator('Title (English)')).to_have_value(jalc_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('掲載誌名 (日本語)')).to_have_value(jalc_expected_journal_ja, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(jalc_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(jalc_expected_date_published, timeout=transition_timeout)
    await expect(form.get_locator('巻')).to_have_value(jalc_expected_volume, timeout=transition_timeout)
    await expect(form.get_locator('号')).to_have_value(jalc_expected_issue, timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (開始)')).to_have_value(jalc_expected_page_start, timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (終了)')).to_have_value(jalc_expected_page_end, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(jalc_expected_author_keyword, timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックしてJaLC補完結果を反映する

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//a[text() = "保存"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、JaLC補完結果の再確認を準備する

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、JaLC補完結果が保持されていることを確認する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('データの名称または論文表題 (日本語)')).to_have_value(jalc_expected_title_ja, timeout=transition_timeout)
    await expect(form.get_locator('Title (English)')).to_have_value(jalc_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('掲載誌名 (日本語)')).to_have_value(jalc_expected_journal_ja, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(jalc_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(jalc_expected_author_keyword, timeout=transition_timeout)

    await page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]').click()
    await expect(page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、PubMed補完の準備をする

同じファイルでPubMedの補完を行うため、再度ファイルを選択し「メタデータ編集」ボタンを有効にする。

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、PubMed補完を開始する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「論文（出版社版）のDOI」にPubMedのDOIを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('論文（出版社版）のDOI', pubmed_doi)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_have_value(pubmed_doi, timeout=transition_timeout)

await run_pw(_step)

## 「PubMedから取得」を選択して補完する

「論文（出版社版）のDOI」欄右側の▼をクリックし、メニューから「PubMedから取得」を選択する。押下後、PubMedから取得したメタデータに置き換わることを確認する。

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = page.locator('//label[contains(text(), "論文（出版社版）のDOI")]/../following-sibling::div[1]')
    dropdown_button = container.locator('button.dropdown-toggle')
    await dropdown_button.click()
    menu_item = container.locator('//ul[contains(@class, \"dropdown-menu\")]//a[text() = "PubMedから取得"]')
    await expect(menu_item).to_be_visible(timeout=transition_timeout)
    await menu_item.click()

    await expect(form.get_locator('Title (English)')).to_have_value(
        pubmed_expected_title_en, timeout=transition_timeout
    )

await run_pw(_step)

## PubMedで自動補完された項目を確認する

- Title (English)
- Journal Name (English)
- 発行年月
- 巻・号
- 掲載ページ (開始/終了)
- 著者名

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(pubmed_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(pubmed_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(pubmed_expected_date_published, timeout=transition_timeout)
    await expect(form.get_locator('巻')).to_have_value(pubmed_expected_volume, timeout=transition_timeout)
    await expect(form.get_locator('号')).to_have_value(pubmed_expected_issue, timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (開始)')).to_have_value(pubmed_expected_page_start, timeout=transition_timeout)
    await expect(form.get_locator('掲載ページ (終了)')).to_have_value(pubmed_expected_page_end, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(pubmed_expected_author_keyword, timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックしてPubMed補完結果を反映する

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//a[text() = "保存"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、PubMed補完結果の再確認を準備する

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、PubMed補完結果が保持されていることを確認する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(pubmed_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('Journal Name (English)')).to_have_value(pubmed_expected_journal_en, timeout=transition_timeout)
    await expect(form.get_locator('著者名')).to_contain_text(pubmed_expected_author_keyword, timeout=transition_timeout)

    await page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]').click()
    await expect(page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、arXiv補完の準備をする

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、arXiv補完を開始する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「論文（出版社版）のDOI」にarXivのDOIを入力する

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('論文（出版社版）のDOI', arxiv_doi)
    await expect(form.get_locator('論文（出版社版）のDOI')).to_have_value(arxiv_doi, timeout=transition_timeout)

await run_pw(_step)

## 「arXivから取得」を選択して補完する

「論文（出版社版）のDOI」欄右側の▼をクリックし、メニューから「arXivから取得」を選択する。押下後、arXivから取得したメタデータに置き換わることを確認する。

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = page.locator('//label[contains(text(), "論文（出版社版）のDOI")]/../following-sibling::div[1]')
    dropdown_button = container.locator('button.dropdown-toggle')
    await dropdown_button.click()
    menu_item = container.locator('//ul[contains(@class, \"dropdown-menu\")]//a[text() = "arXivから取得"]')
    await expect(menu_item).to_be_visible(timeout=transition_timeout)
    await menu_item.click()

    await expect(form.get_locator('Title (English)')).to_have_value(
        arxiv_expected_title_en, timeout=transition_timeout
    )

await run_pw(_step)

## arXivで自動補完された項目を確認する

- Title (English)
- 発行年月 (取得できる場合)
- 著者名

注: Journal Name (English) はarXiv補完では設定されないため確認対象外

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(arxiv_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(arxiv_expected_date_published, timeout=transition_timeout)

    await expect(form.get_locator('著者名')).to_contain_text(arxiv_expected_author_keyword, timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックしてarXiv補完結果を反映する

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//a[text() = "保存"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テスト用ファイルをクリックし、arXiv補完結果の再確認を準備する

In [ ]:
async def _step(page):
    if not use_file_view:
        await grdm.get_select_file_extension_locator(page, crossref_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックし、arXiv補完結果が保持されていることを確認する

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    form = FileMetadataForm(page)
    await expect(form.get_locator('Title (English)')).to_have_value(arxiv_expected_title_en, timeout=transition_timeout)
    await expect(form.get_locator('発行年月')).to_have_value(arxiv_expected_date_published, timeout=transition_timeout)

    await expect(form.get_locator('著者名')).to_contain_text(arxiv_expected_author_keyword, timeout=transition_timeout)

    await page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]').click()
    await expect(page.locator('//*[@class = "modal-footer"]//a[text() = "閉じる"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 後始末

必要に応じてプロジェクトを削除する

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await grdm.delete_project(page, transition_timeout=transition_timeout)
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()